# Train the aspects-scoring ML detector on Kaggle (GPU)

Self-contained -- no dependency on the aspects-scoring repo being present, so it runs from a fresh Kaggle notebook.

**Before running:**
1. Upload `data/aisd_nifti/` (605MB) and `data/split.json` from your local `aspects-scoring` repo as a new Kaggle Dataset (keep the same folder structure: one folder per patient ID, each containing `image.nii.gz` and `mask_multiclass.nii.gz`, plus `split.json` at the top level).
2. Attach that dataset to this notebook (Add Data, on the right sidebar).
3. Settings -> Accelerator -> GPU (T4 x2 or P100).
4. Fix `DATA_DIR` below to match your dataset's actual mounted path (visible under `/kaggle/input/<your-dataset-slug>/`).
5. Run all cells. The best checkpoint saves to `/kaggle/working/best_ml_detector.pt` -- download it from the notebook's Output/Files panel when done, and bring it back to `aspects-scoring/models/best_ml_detector.pt` locally.

In [ ]:
import json
import time
from collections import OrderedDict
from pathlib import Path

import numpy as np
import nibabel as nib
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from scipy import ndimage

# ADJUST THIS to your attached dataset's mounted path.
DATA_DIR = Path("/kaggle/input/aisd-nifti/aisd_nifti")
SPLIT_PATH = Path("/kaggle/input/aisd-nifti/split.json")
OUTPUT_DIR = Path("/kaggle/working")

IMAGE_SIZE = 256
VISIBLE_ON_CT_LABELS = {1, 2, 3, 5}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Preprocessing (inlined from aspects-scoring/src/preprocessing.py and detection.py -- kept minimal, no rotation-search here, same tradeoff as the local script: rotation correction matters for final inference quality, not for training useful features, and would slow data loading a lot)

In [ ]:
def skull_strip_threshold(volume, hu_low=20, hu_high=230):
    mask = (volume >= hu_low) & (volume <= hu_high)
    labeled, n = ndimage.label(mask)
    if n == 0:
        return mask
    sizes = ndimage.sum(mask, labeled, range(1, n + 1))
    largest = np.argmax(sizes) + 1
    brain_mask = labeled == largest
    brain_mask = ndimage.binary_fill_holes(brain_mask)
    return ndimage.binary_closing(brain_mask, iterations=2)


def brain_center_x(brain_mask):
    return np.argwhere(brain_mask)[:, 0].mean()


def mirror_across_x(volume, brain_mask):
    center_x = brain_center_x(brain_mask)
    flipped = volume[::-1, :, :]
    shift = 2 * center_x - (volume.shape[0] - 1)
    return ndimage.shift(flipped, shift=(shift, 0, 0), order=1, mode="nearest")


def load_patient_arrays(patient_id):
    image_path = DATA_DIR / patient_id / "image.nii.gz"
    mask_path = DATA_DIR / patient_id / "mask_multiclass.nii.gz"
    raw = nib.load(str(image_path)).get_fdata(dtype=np.float32)
    windowed = np.clip(raw, 0, 255) / 255.0
    brain_mask = skull_strip_threshold(raw)
    mirrored = mirror_across_x(windowed, brain_mask)
    diff = np.abs(windowed - mirrored)
    diff[~brain_mask] = 0

    target = None
    if mask_path.exists():
        mask_raw = nib.load(str(mask_path)).get_fdata(dtype=np.float32)
        target = np.isin(mask_raw, list(VISIBLE_ON_CT_LABELS)).astype(np.float32)
    return windowed, mirrored, diff, target

## Model (SmallUNet, matches src/ml_detector.py) + Dataset

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.InstanceNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.InstanceNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.down1 = DoubleConv(3, 16)
        self.down2 = DoubleConv(16, 32)
        self.down3 = DoubleConv(32, 64)
        self.pool = nn.MaxPool2d(2)
        self.up2 = DoubleConv(64 + 32, 32)
        self.up1 = DoubleConv(32 + 16, 16)
        self.head = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        c1 = self.down1(x)
        c2 = self.down2(self.pool(c1))
        c3 = self.down3(self.pool(c2))
        u2 = nn.functional.interpolate(c3, size=c2.shape[-2:], mode="bilinear", align_corners=False)
        u2 = self.up2(torch.cat([u2, c2], dim=1))
        u1 = nn.functional.interpolate(u2, size=c1.shape[-2:], mode="bilinear", align_corners=False)
        u1 = self.up1(torch.cat([u1, c1], dim=1))
        return self.head(u1)


def dice_loss(logits, target, eps=1.0):
    probs = torch.sigmoid(logits)
    inter = (probs * target).sum(dim=(1, 2, 3))
    denom = probs.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return 1.0 - ((2 * inter + eps) / (denom + eps)).mean()


def combined_loss(logits, target, pos_weight):
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target, pos_weight=pos_weight)
    return bce + dice_loss(logits, target)


class AISDSliceDataset(Dataset):
    def __init__(self, patient_ids, cache_limit=4):
        self.slices = []
        for pid in patient_ids:
            mask_path = DATA_DIR / pid / "mask_multiclass.nii.gz"
            if not mask_path.exists():
                continue
            n_slices = nib.load(str(mask_path)).shape[2]
            for z in range(n_slices):
                self.slices.append((pid, z))
        self._cache = OrderedDict()
        self._cache_limit = cache_limit

    def __len__(self):
        return len(self.slices)

    def _get_patient(self, pid):
        if pid not in self._cache:
            self._cache[pid] = load_patient_arrays(pid)
            while len(self._cache) > self._cache_limit:
                self._cache.popitem(last=False)
        else:
            self._cache.move_to_end(pid)
        return self._cache[pid]

    def __getitem__(self, idx):
        pid, z = self.slices[idx]
        windowed, mirrored, diff, target = self._get_patient(pid)
        x = np.stack([windowed[:, :, z], mirrored[:, :, z], diff[:, :, z]], axis=0).astype(np.float32)
        y = target[:, :, z][None, ...]
        x_t = torch.from_numpy(x).unsqueeze(0)
        y_t = torch.from_numpy(y).unsqueeze(0)
        x_t = nn.functional.interpolate(x_t, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)
        y_t = nn.functional.interpolate(y_t, size=(IMAGE_SIZE, IMAGE_SIZE), mode="nearest")
        return x_t.squeeze(0), y_t.squeeze(0)

## Load split, build loaders

In [ ]:
split = json.loads(SPLIT_PATH.read_text())
train_ids = split["train"]
val_ids = split["validation"]
print(f"train patients: {len(train_ids)}, validation patients: {len(val_ids)}")

train_ds = AISDSliceDataset(train_ids)
val_ds = AISDSliceDataset(val_ids)
print(f"train slices: {len(train_ds)}, validation slices: {len(val_ds)}")

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## Estimate class imbalance for pos_weight

In [ ]:
sample_pos, sample_total = 0.0, 0.0
for i in range(min(50, len(train_ds))):
    _, y = train_ds[i]
    sample_pos += float(y.sum())
    sample_total += float(y.numel())
pos_frac = max(sample_pos / sample_total, 1e-4)
pos_weight = torch.tensor([min((1 - pos_frac) / pos_frac, 100.0)]).to(device)
print(f"positive-voxel fraction: {pos_frac:.5f}, pos_weight: {pos_weight.item():.2f}")

## Train

In [ ]:
def dice_from_logits(logits, target):
    pred = (torch.sigmoid(logits) > 0.5).float()
    inter = (pred * target).sum(dim=(1, 2, 3))
    denom = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return ((2 * inter + 1.0) / (denom + 1.0)).mean()


def run_epoch(model, loader, optimizer, training):
    model.train(training)
    total_loss, total_dice, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.set_grad_enabled(training):
            logits = model(x)
            loss = combined_loss(logits, y, pos_weight)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item()
        total_dice += dice_from_logits(logits.detach(), y).item()
        n += 1
    return total_loss / max(n, 1), total_dice / max(n, 1)


EPOCHS = 15
model = SmallUNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_val_dice = -1.0
history = []
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_dice = run_epoch(model, train_loader, optimizer, training=True)
    val_loss, val_dice = run_epoch(model, val_loader, optimizer, training=False)
    elapsed = time.time() - t0
    row = {"epoch": epoch, "train_loss": train_loss, "train_dice": train_dice,
           "val_loss": val_loss, "val_dice": val_dice, "seconds": elapsed}
    history.append(row)
    print(row)
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), OUTPUT_DIR / "best_ml_detector.pt")
        print(f"  -> new best (val_dice={val_dice:.4f}), checkpoint saved")

(OUTPUT_DIR / "history.json").write_text(json.dumps(history, indent=2))
print(f"\nBest validation Dice: {best_val_dice:.4f}")
print("Download /kaggle/working/best_ml_detector.pt and bring it back to aspects-scoring/models/")